# 03 — TF-IDF + logistic regression baseline

A simple, transparent model to compare the four pretrained classifiers against.

- Trained on `baseline_train` (WILDS train, 269,038 comments).
- Tuned on `baseline_val` (WILDS val, 45,180 comments).
- Scored on `audit_tune`, `audit_test`, and HateCheck, the same texts as the pretrained models. The scores are saved next to theirs on the scores folder.
- Feature analysis: because the model is linear, we can read which words push a comment toward "toxic", including identity words.

In [ ]:
%pip install -q pandas pyarrow scikit-learn

In [ ]:
import os

# --- Colab + Google Drive setup (skipped automatically when running locally) ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/FA26 classes/CS74 - Sarah Preum/FinalProject/"
    os.chdir(PROJECT_DIR)
except ImportError:
    pass  # not on Colab: run from the project folder
print("working directory:", os.getcwd())

Mounted at /content/drive
working directory: /content/drive/MyDrive/FA26 classes/CS74 - Sarah Preum/FinalProject


In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

SEED = 42
PROC = Path("data/processed")
SCORES = Path("scores")
(SCORES / "per_model").mkdir(parents=True, exist_ok=True)
NAME = "TF-IDF LR"   # column name used next to the pretrained models' scores

print("scikit-learn", sklearn.__version__)

scikit-learn 1.6.1


In [ ]:
train = pd.read_parquet(PROC / "baseline_train.parquet")
val = pd.read_parquet(PROC / "baseline_val.parquet")
for df in (train, val):
    df["comment_text"] = df["comment_text"].fillna("")   # 2 comments have missing text
print(f"train: {len(train):,} ({100*train['toxic'].mean():.1f}% toxic)")
print(f"val:   {len(val):,} ({100*val['toxic'].mean():.1f}% toxic)")

train: 269,038 (11.3% toxic)
val:   45,180 (11.2% toxic)


## Model and hyperparameter search

**Features (TF-IDF):** each comment becomes a vector with one value per word (or word pair). A word gets a high value if it appears often in *this* comment but is rare across all comments. Settings:
- `lowercase=True` (default): "Idiot" and "idiot" count as the same word. Simple baselines usually do this; the transformers don't need it.
- `min_df=3`: ignore words that appear in fewer than 5 comments (typos, names).
- `max_features=50,000`: keep the 50k most frequent words/pairs, to keep it fast.
- `sublinear_tf=True`: use 1 + log(count) instead of the raw count, so a word repeated 10 times doesn't count 10× as much.

**Classifier:** logistic regression with `class_weight="balanced"`. Only ~11% of comments are toxic, so without this the model would lean toward predicting "not toxic". Balancing -- give the toxic examples more weight in training.

**Grid** (6 combinations), chosen by ROC-AUC on `baseline_val`:
- `ngram_range`: single words `(1, 1)` vs. single words + word pairs `(1, 2)`. Pairs can catch phrases like "not bad".
- `C` (inverse regularization strength): `0.1`, `1`, `10`. Small C = simpler model (more regularization); large C = fits the training data more closely.

In [ ]:
GRID_NGRAMS = [(1, 1), (1, 2)]
GRID_C = [0.1, 1, 10]

results = []
fitted = {}
for ngram in GRID_NGRAMS:
    # The vectorizer is fit on TRAIN only, so nothing about val leaks into the features
    vec = TfidfVectorizer(ngram_range=ngram, min_df=5, max_features=50_000, sublinear_tf=True)
    X_train = vec.fit_transform(train["comment_text"])
    X_val = vec.transform(val["comment_text"])
    for C in GRID_C:
        t0 = time.time()
        clf = LogisticRegression(C=C, class_weight="balanced", solver="liblinear",
                                 max_iter=1000, random_state=SEED)
        clf.fit(X_train, train["toxic"])
        auc = roc_auc_score(val["toxic"], clf.predict_proba(X_val)[:, 1])
        results.append({"ngram_range": ngram, "C": C, "val_auc": auc,
                        "fit_seconds": round(time.time() - t0, 1)})
        fitted[(ngram, C)] = (vec, clf)
        print(f"ngram={ngram} C={C:<4} val AUC={auc:.4f}")

grid = pd.DataFrame(results).sort_values("val_auc", ascending=False)
grid.to_csv(SCORES / "baseline_grid.csv", index=False)
grid

ngram=(1, 1) C=0.1  val AUC=0.9079
ngram=(1, 1) C=1    val AUC=0.9215
ngram=(1, 1) C=10   val AUC=0.9072
ngram=(1, 2) C=0.1  val AUC=0.8983
ngram=(1, 2) C=1    val AUC=0.9132
ngram=(1, 2) C=10   val AUC=0.8928


,ngram_range,C,val_auc,fit_seconds
1,"(1, 1)",1.0,0.921533,9.4
4,"(1, 2)",1.0,0.913173,10.6
0,"(1, 1)",0.1,0.907872,5.3
2,"(1, 1)",10.0,0.907219,23.2
3,"(1, 2)",0.1,0.898331,6.8
5,"(1, 2)",10.0,0.892833,22.5


In [ ]:
best = grid.iloc[0]
best_key = (best["ngram_range"], best["C"])
vec, clf = fitted[best_key]
print(f"best: ngram_range={best_key[0]}, C={best_key[1]}  (val AUC {best['val_auc']:.4f})")

best: ngram_range=(1, 1), C=1.0  (val AUC 0.9215)


## Score the same texts as the pretrained models

The baseline is **not** refit on `audit_tune`; it's used exactly as selected on `baseline_val`. Its scores are saved in the same format as the pretrained models' scores, and added as a `TF-IDF LR` column to the combined files from `02_score_models.ipynb`.

In [ ]:
DATASETS = {
    # name: (file, id column, text column, label column)
    "audit_tune": (PROC / "audit_tune.parquet", "id", "comment_text", "toxic"),
    "audit_test": (PROC / "audit_test.parquet", "id", "comment_text", "toxic"),
    "hatecheck":  (PROC / "hatecheck.parquet", "case_id", "test_case", "hateful"),
}

check = {}
for ds, (path, id_col, text_col, label_col) in DATASETS.items():
    df = pd.read_parquet(path)
    s = clf.predict_proba(vec.transform(df[text_col].fillna("")))[:, 1]
    out = pd.DataFrame({"id": df[id_col], "score": s.astype("float32")})
    out.to_parquet(SCORES / "per_model" / f"{ds}__TF-IDF_LR.parquet")
    check[ds] = roc_auc_score(df[label_col], s)

    # Add / replace the column in the combined file from notebook 02 (safe to re-run)
    combined_path = SCORES / f"{ds}.parquet"
    if combined_path.exists():
        combined = pd.read_parquet(combined_path).drop(columns=[NAME], errors="ignore")
        combined = combined.merge(out.rename(columns={"score": NAME}), on="id", how="left")
        assert combined[NAME].notna().all(), f"{ds}: some texts are missing a baseline score"
        combined.to_parquet(combined_path)
        print(f"{ds}: {len(out):,} scored, column '{NAME}' added to {combined_path}")
    else:
        print(f"{ds}: {len(out):,} scored (no {combined_path} yet; run 02 first, then re-run this cell)")

with open(SCORES / "baseline_info.json", "w") as f:
    json.dump({"ngram_range": list(best_key[0]), "C": float(best_key[1]),
               "min_df": 5, "max_features": 50_000, "sublinear_tf": True,
               "class_weight": "balanced", "solver": "liblinear", "seed": SEED,
               "val_auc": float(best["val_auc"]), "sklearn": sklearn.__version__}, f, indent=2)

audit_tune: 21,148 scored, column 'TF-IDF LR' added to scores/audit_tune.parquet
audit_test: 21,147 scored, column 'TF-IDF LR' added to scores/audit_test.parquet
hatecheck: 3,728 scored, column 'TF-IDF LR' added to scores/hatecheck.parquet


**Smoke check only.** The real evaluation is on `audit_test` in `04_experiments.ipynb`. Expect a lower HateCheck AUC: a bag-of-words model can't understand negation or counter-speech.

In [ ]:
pd.Series(check, name="ROC-AUC").round(3)

,ROC-AUC
audit_tune,0.919
audit_test,0.920
hatecheck,0.578


## Feature analysis: which words does the baseline rely on?

Each word has one **coefficient** (weight). Positive = pushes the comment toward "toxic", negative = toward "not toxic". This is direct evidence of *feature reliance*: if identity words like "gay" or "muslim" get positive weights, the model has learned the shortcut "identity word → toxic" from the data (Table 1 in the report shows why: comments that mention these groups are labeled toxic 2–3× more often).

In [ ]:
coefs = pd.Series(clf.coef_[0], index=vec.get_feature_names_out()).sort_values()
print("Most NON-toxic words:", ", ".join(coefs.head(20).index))
print()
print("Most toxic words:    ", ", ".join(coefs.tail(20).index[::-1]))

Most NON-toxic words: differences, april, incorrect, secular, http, gdp, challenging, gained, approval, individuals, orientation, classes, factor, spectrum, india, includes, francis, named, june, experienced

Most toxic words:     stupid, idiot, idiots, stupidity, pathetic, ignorant, dumb, crap, ridiculous, idiotic, hypocrite, damn, fool, fools, silly, garbage, moron, ass, jerk, hypocrites


In [ ]:
# Identity words: where do they fall among all 50k words?
IDENTITY_WORDS = ["gay", "lesbian", "homosexual", "transgender", "straight",
                  "muslim", "muslims", "islam", "christian", "christians", "jewish", "jews",
                  "black", "blacks", "white", "whites", "african",
                  "woman", "women", "female", "man", "men", "male"]
rank_pct = coefs.rank(pct=True)   # 1.0 = the most toxic-weighted word
ident = pd.DataFrame({
    "coefficient": coefs.reindex(IDENTITY_WORDS),
    "percentile (100 = most toxic)": (100 * rank_pct.reindex(IDENTITY_WORDS)),
}).dropna().sort_values("coefficient", ascending=False).round(2)
ident.to_csv(SCORES / "baseline_identity_coefs.csv")
ident

,coefficient,percentile (100 = most toxic)
gay,10.73,99.95
black,9.40,99.91
blacks,8.38,99.90
white,8.26,99.90
muslims,7.27,99.86
homosexual,7.27,99.86
muslim,6.62,99.83
transgender,5.44,99.75
whites,5.19,99.72
lesbian,4.94,99.69


--> a coefficient above 0 means that word alone makes the baseline more likely to call a comment toxic, even in a neutral sentence like "I am a gay man." A high percentile (e.g. 95+) means the word is among the most toxic-weighted words in the whole vocabulary. Compare with the subgroup results of the pretrained models in `04_experiments.ipynb`.